# Stage 2 — Instruction (SFT) Fine-Tuning

**Environment:** Lightning.ai Studio, T4 GPU (16GB VRAM). **Base checkpoint:** Stage 1's merged model (`unsloth/Llama-3.2-3B` + continued pretraining), loaded from the Teamspace path or the shared HF Hub repo's `stage1-merged/` folder.

Differences from Stage 1, per the project's Stage 1 vs Stage 2 design:

| Parameter | Stage 1 | Stage 2 |
|---|---|---|
| Data format | Raw paragraphs (`text`) | Formatted `### Instruction: ... ### Response: ...` (`text`) |
| `packing` | `True` | `False` (preserves prompt/response boundaries) |
| Learning rate | `2e-4` | `1e-4` (lower, safer for instruction behavior) |
| Loss masking | N/A | **Unmasked** — loss computed on the full instruction+response sequence (a deliberate choice here; prompt-masked loss, which trains only on the response tokens, is the more standard approach and usually yields sharper instruction-following if Stage 2's eval answers look weak) |

This notebook:
1. Loads `data/instruction_dataset.jsonl` (100 instruction/response pairs) and formats each into an Alpaca-style `text` field
2. Loads Stage 1's merged model in 4-bit and attaches a fresh LoRA adapter
3. Runs SFT with `SFTTrainer`/`SFTConfig` (`packing=False`)
4. Saves the adapter and merged model to the Teamspace path, then uploads both to the shared HF Hub repo under `stage2-adapter/` and `stage2-merged/`
5. Reloads the merged model and runs the same 10 evaluation questions, now in instruction format, for the `reports/sft_model_comparison.md` table

## 1. Install libraries

In [1]:
!pip -q install unsloth
!pip -q install transformers==4.56.2
!pip -q install --no-deps trl==0.22.2
!pip -q install -U pymupdf datasets
!pip -q install huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.6.9 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.
unsloth-zoo 2026.6.7 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.0 which is incompatible.


## 2. Imports

In [1]:
import os
import gc
import time
import json
import warnings
from typing import List, Dict, Any

warnings.filterwarnings("ignore")

import torch
from datasets import Dataset

import unsloth  # keep this import early
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

from huggingface_hub import HfApi, notebook_login

assert torch.cuda.is_available(), "GPU not found. On Lightning.ai: select a GPU-backed Studio (e.g. T4) before running."
print("GPU:", torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4


## 3. Config
`HF_REPO` and `LIGHTNING_ROOT` must match the values used in `non_instruction_finetuning.ipynb` so this notebook finds Stage 1's outputs.

In [2]:
# ---- File paths ----
instruction_data_path = "data/instruction_dataset.jsonl"

# ---- Model ----
MAX_SEQ_LENGTH = 512
SEED = 4212

# ---- QLoRA hyperparameters ----
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
WARMUP_STEPS = 10
LOGGING_STEPS = 5
NUM_TRAIN_EPOCHS = 20

STAGE2_LR = 1e-4  # lower than Stage 1's 2e-4, per the project's Stage 1 vs Stage 2 design

# ---- Persistent storage (Lightning.ai Teamspace path; the ONLY local-disk write) ----
LIGHTNING_ROOT = "/teamspace/studios/this_studio/hummingbird-assistant-llm"
STAGE1_MERGED_DIR  = f"{LIGHTNING_ROOT}/stage1-merged"   # Stage 2's starting checkpoint
STAGE2_ADAPTER_DIR = f"{LIGHTNING_ROOT}/stage2-adapter"
STAGE2_MERGED_DIR  = f"{LIGHTNING_ROOT}/stage2-merged"

# ---- Same shared HF repo used across all three stages ----
HF_REPO = "Hummingbird-assistant-llm"

# If Stage 1's merged model isn't found locally in the Teamspace path, fall back to
# downloading it from the shared HF repo's stage1-merged/ subfolder instead.
HF_STAGE1_MERGED_SUBFOLDER = "stage1-merged"

## 4. Resolve Stage 1's checkpoint, set up storage, and log in to Hugging Face Hub
Tries the local Teamspace path first (same Studio as Stage 1); if that's not found (e.g. a fresh Studio), downloads Stage 1's merged model from the shared HF repo instead.

In [3]:
os.makedirs(STAGE2_ADAPTER_DIR, exist_ok=True)
os.makedirs(STAGE2_MERGED_DIR, exist_ok=True)

notebook_login()  # paste a Hugging Face write token when prompted

hf_api = HfApi()
#hf_api.create_repo(repo_id=HF_REPO, exist_ok=True, repo_type="model")
repo_info = hf_api.create_repo(repo_id=HF_REPO, exist_ok=True, repo_type="model")
HF_REPO_FULL = repo_info.repo_id  # namespaced id, e.g. "Snow79703/Hummingbird-assistant-llm"

if os.path.exists(STAGE1_MERGED_DIR) and os.listdir(STAGE1_MERGED_DIR):
    stage2_base_model_path = STAGE1_MERGED_DIR
    print(f"Using Stage 1 merged model from Teamspace: {stage2_base_model_path}")
else:
    from huggingface_hub import snapshot_download
    print("Stage 1 merged model not found locally -- downloading from HF Hub instead...")
    stage2_base_model_path = snapshot_download(
        repo_id=HF_REPO,
        allow_patterns=f"{HF_STAGE1_MERGED_SUBFOLDER}/*",
    )
    stage2_base_model_path = os.path.join(stage2_base_model_path, HF_STAGE1_MERGED_SUBFOLDER)
    print(f"Downloaded Stage 1 merged model to: {stage2_base_model_path}")

Using Stage 1 merged model from Teamspace: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage1-merged


## 5. Helper functions
Identical to `non_instruction_finetuning.ipynb`, with one addition: `generate_answer`, which wraps a question in the Alpaca-style instruction prompt used for training.

In [4]:
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()


def train_and_measure(trainer, stage_name: str):
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

    start_time = time.time()
    result = trainer.train()
    torch.cuda.synchronize()

    train_time = round(time.time() - start_time, 2)
    peak_allocated = round(torch.cuda.max_memory_allocated() / 1024**3, 3)
    peak_reserved = round(torch.cuda.max_memory_reserved() / 1024**3, 3)

    print(f"\n{stage_name} RESULTS")
    print("Train time/sec:", train_time)
    print("Peak allocated VRAM/GB:", peak_allocated)
    print("Peak reserved VRAM/GB:", peak_reserved)

    return result


def build_instruction_prompt(instruction: str, response: str = None) -> str:
    """Alpaca-style prompt. When response is None, returns just the prompt
    portion (used at inference time); when provided, appends the response
    plus an EOS marker (used to build training examples)."""
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    if response is None:
        return prompt
    return prompt + response


def generate_answer(model, tokenizer, instruction: str, max_new_tokens: int = 200):
    """Instruction-formatted generation -- used from Stage 2 onward, now that
    the model has been trained to expect the ### Instruction / ### Response format."""
    FastLanguageModel.for_inference(model)

    prompt = build_instruction_prompt(instruction)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_tokens = inputs["input_ids"].shape[-1]
    generated_tokens = output[0][input_tokens:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def load_unsloth_model_with_lora(model_name_or_path: str):
    """Loads a base or merged model in 4-bit and attaches a fresh LoRA adapter."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name_or_path,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    model.print_trainable_parameters()
    return model, tokenizer


def save_adapter_and_merge(model, tokenizer, adapter_dir: str, merged_dir: str,
                            hf_repo: str, stage_prefix: str, stage_name: str,
                            base_model_id: str):
    """Saves the LoRA adapter + merged 16-bit model to the Teamspace path,
    then uploads both to the shared hf_repo under per-stage subfolders.

    base_model_id must be a real Hugging Face Hub id (not a local path) --
    it gets written into the adapter's README.md metadata, and the Hub
    rejects local paths there on upload.
    """
    def _fix_readme_base_model(readme_path: str, base_model_id: str):
        """Force-corrects the 'base_model' field in a README's YAML front
        matter to a real Hub id. Needed because Unsloth's save routines
        generate this README and leak the local load path into it,
        regardless of peft_config."""
        if not os.path.exists(readme_path):
            return
        with open(readme_path, "r", encoding="utf8") as f:
            content = f.read()
        if not content.startswith("---"):
            return
        end = content.find("\n---", 3)
        if end == -1:
            return
        frontmatter_lines = content[3:end].split("\n")
        found = False
        for i, line in enumerate(frontmatter_lines):
            if line.strip().startswith("base_model:"):
                frontmatter_lines[i] = f"base_model: {base_model_id}"
                found = True
        if not found:
            frontmatter_lines.append(f"base_model: {base_model_id}")
        new_content = "---" + "\n".join(frontmatter_lines) + content[end:]
        with open(readme_path, "w", encoding="utf8") as f:
            f.write(new_content)

    print(f"\nSaving {stage_name} adapter to {adapter_dir} ...")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    _fix_readme_base_model(os.path.join(adapter_dir, "README.md"), base_model_id)
    print(f"{stage_name} adapter saved to: {adapter_dir}")

    print(f"\nMerging {stage_name} adapter with base model, saving merged model to {merged_dir} ...")
    FastLanguageModel.for_training(model)
    model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
    _fix_readme_base_model(os.path.join(merged_dir, "README.md"), base_model_id)
    print(f"{stage_name} merged model saved to: {merged_dir}")

    print(f"\nUploading {stage_name} adapter to https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-adapter ...")
    hf_api.upload_folder(
        repo_id=hf_repo,
        folder_path=adapter_dir,
        path_in_repo=f"{stage_prefix}-adapter",
        repo_type="model",
    )

    print(f"Uploading {stage_name} merged model to https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-merged ...")
    hf_api.upload_folder(
        repo_id=hf_repo,
        folder_path=merged_dir,
        path_in_repo=f"{stage_prefix}-merged",
        repo_type="model",
    )

    print(f"\n{stage_name} complete. Teamspace: {adapter_dir}, {merged_dir}")
    print(f"HF Hub: https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-adapter")
    print(f"HF Hub: https://huggingface.co/{hf_repo}/tree/main/{stage_prefix}-merged")

## 6. Stage 2 data: load and format the instruction dataset
Each `{instruction, response}` pair from `data/instruction_dataset.jsonl` is formatted into a single Alpaca-style `text` field, with the tokenizer's EOS token appended so the model learns where a response should end.

In [5]:
def build_instruction_dataset(path: str) -> Dataset:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Add this file to the Studio first -- "
            f"e.g. clone the repo, drag-and-drop via the file browser, or scp it in."
        )

    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))

    if len(records) == 0:
        raise ValueError("No usable records found in the instruction dataset file.")

    print("Instruction/response records:", len(records))
    print("\nSample record:", records[0])

    texts = [
        build_instruction_prompt(r["instruction"], r["response"]) + tokenizer.eos_token
        for r in records
    ]
    print("\nSample formatted training text:\n", texts[0])

    return Dataset.from_dict({"text": texts})

## 7. Stage 2: Instruction (SFT) fine-tuning
Starts from Stage 1's merged model. `packing=False` here (unlike Stage 1) to preserve each example's instruction/response boundary rather than concatenating multiple examples into one packed sequence.

In [6]:
print("\n==============================")
print("STAGE 2: INSTRUCTION (SFT) TRAINING")
print("==============================")

clear_gpu_memory()
stage2_model, tokenizer = load_unsloth_model_with_lora(stage2_base_model_path)

stage2_dataset = build_instruction_dataset(instruction_data_path)

FastLanguageModel.for_training(stage2_model)

stage2_config = SFTConfig(
    output_dir="/tmp/stage2_logs",  # ephemeral trainer logs only, not a deliverable

    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    learning_rate=STAGE2_LR,
    warmup_steps=WARMUP_STEPS,

    logging_steps=LOGGING_STEPS,
    save_strategy="no",
    report_to="none",

    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",

    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,  # preserve instruction/response boundaries -- do not concatenate examples

    seed=SEED,
)

stage2_trainer = SFTTrainer(
    model=stage2_model,
    processing_class=tokenizer,
    train_dataset=stage2_dataset,
    args=stage2_config,
)

train_and_measure(stage2_trainer, "STAGE 2 - INSTRUCTION (SFT) TRAINING")


STAGE 2: INSTRUCTION (SFT) TRAINING
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
Instruction/response records: 100

Sample record: {'instruction': 'How do hummingbirds achieve hovering flight unlike most other birds?', 'response': "Hummingbirds hover by rotating their wings almost 180 degrees at the shoulder, generating lift on both the forward and backward stroke rather than only the downstroke as in typical bird flight. This figure-eight wingbeat pattern lets them produce nearly equal lift throughout the wingbeat cycle. Their unique shoulder joint anatomy, with a highly mobile ball-and-socket structure, permits this degree of rotation that most other birds' wings cannot achieve."}

Sample formatted training text:
 ### Instruction:
How do hummingbirds achieve hovering flight unlike most other birds?

### Response:
Hummingbirds hover by rotating their wings almost 180 degrees at the shoulder, generating lift on both the forward and backward stroke rather than only the downstroke as in t

Unsloth: Tokenizing ["text"] (num_proc=7):   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 20 | Total steps = 260
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
5,2.440300
10,2.278600
15,1.943300
20,1.793800
25,1.738000
30,1.558900
35,1.465700
40,1.342500
45,1.155600
50,1.132100



STAGE 2 - INSTRUCTION (SFT) TRAINING RESULTS
Train time/sec: 532.5
Peak allocated VRAM/GB: 2.611
Peak reserved VRAM/GB: 2.785


TrainOutput(global_step=260, training_loss=0.4602621911236873, metrics={'train_runtime': 530.7105, 'train_samples_per_second': 3.769, 'train_steps_per_second': 0.49, 'total_flos': 3725888626728960.0, 'train_loss': 0.4602621911236873, 'epoch': 20.0})

## 8. Save adapter, merge, and upload (Teamspace + shared HF Hub repo, subfoldered)

In [7]:
save_adapter_and_merge(
    model=stage2_model,
    tokenizer=tokenizer,
    adapter_dir=STAGE2_ADAPTER_DIR,
    merged_dir=STAGE2_MERGED_DIR,
    hf_repo=HF_REPO_FULL,
    stage_prefix="stage2",
    stage_name="Stage 2",
    base_model_id=HF_REPO_FULL,
)

del stage2_trainer
del stage2_model
clear_gpu_memory()


Saving Stage 2 adapter to /teamspace/studios/this_studio/hummingbird-assistant-llm/stage2-adapter ...
Stage 2 adapter saved to: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage2-adapter

Merging Stage 2 adapter with base model, saving merged model to /teamspace/studios/this_studio/hummingbird-assistant-llm/stage2-merged ...
Detected local model directory: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage1-merged
Found HuggingFace hub cache directory: /teamspace/studios/this_studio/.cache/huggingface/hub


Unsloth: Preparing safetensor model files:  50%|█████     | 1/2 [00:02<00:02,  2.33s/it]

Copied model-00002-of-00002.safetensors from local model directory


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:18<00:00,  9.26s/it]


Copied model-00001-of-00002.safetensors from local model directory


...

New Data Upload: |          |  0.00B /  0.00B            

Uploading Stage 2 merged model to https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage2-merged ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Stage 2 complete. Teamspace: /teamspace/studios/this_studio/hummingbird-assistant-llm/stage2-adapter, /teamspace/studios/this_studio/hummingbird-assistant-llm/stage2-merged
HF Hub: https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage2-adapter
HF Hub: https://huggingface.co/Snow79703/Hummingbird-assistant-llm/tree/main/stage2-merged


## 9. Inference demo (on the freshly reloaded MERGED model, instruction format)

Loads the just-saved merged model fresh from the Teamspace path and runs the same 10 questions as Stage 1 and the base model evaluation — now wrapped in the Alpaca instruction format, since the model has been trained to expect it. This should show a clear improvement in directness and coherence over both the raw base model and the Stage 1 (non-instruction) output.

In [8]:
inference_model, inference_tokenizer = FastLanguageModel.from_pretrained(
    model_name=STAGE2_MERGED_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(inference_model)

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm

In [9]:
eval_questions = [
    "What is your knowledge cutoff date?",
    "What is a hummingbird's heart rate, and how does it change during flight?",
    "How fast can hummingbirds fly, including during a courtship dive?",
    "What is torpor in hummingbirds and why do they use it?",
    "How far do rufous hummingbirds migrate each year?",
    "What is the typical wingbeat frequency of a hovering hummingbird?",
    "How many eggs does a female hummingbird lay per clutch?",
    "What is the smallest hummingbird species in the world?",
    "What are the main threats to hummingbird populations today?",
    "Why can hummingbirds hover while most other birds cannot?",
    "What adaptations allow hummingbirds to consume nectar and process it so efficiently?"
]

stage2_results = []
for i, q in enumerate(eval_questions, start=1):
    answer = generate_answer(inference_model, inference_tokenizer, q, max_new_tokens=200)
    stage2_results.append({"id": i, "question": q, "stage2_answer": answer})
    print(f"Q{i}: {q}")
    print(f"A{i}: {answer}")
    print("-" * 80)

Q1: What is your knowledge cutoff date?
A1: I generally use a knowledge cutoff date of about two months prior to the exam, since newly published research can potentially confuse students by being too recent to be considered established but still included on the test. This cutoff date ensures that recent high-profile studies or major taxonomic revisions are not unduly influencing exam questions without having had sufficient time for widespread acceptance among the larger ornithological community.
--------------------------------------------------------------------------------
Q2: What is a hummingbird's heart rate, and how does it change during flight?
A2: During sustained hovering, a hummingbird's heart rate can exceed 1,200 to 1,300 beats per minute, some of the highest heart rates of any vertebrate. This extreme cardiovascular demand is necessary to generate the continuous blood flow needed to fuel the high muscle contraction demands of hovering flight. Heart rate typically drops dur

### Markdown table (paste-ready for `reports/sft_model_comparison.md`)

In [10]:
def to_markdown_table(results, answer_key, header):
    lines = [f"| # | Question | {header} |", "|---|---|---|"]
    for r in results:
        q = r["question"].replace("|", "\\|")
        a = r[answer_key].replace("\n", " ").replace("|", "\\|")
        lines.append(f"| {r['id']} | {q} | {a} |")
    return "\n".join(lines)


print(to_markdown_table(stage2_results, "stage2_answer", "Stage 2 (Instruction/SFT) Answer"))

| # | Question | Stage 2 (Instruction/SFT) Answer |
|---|---|---|
| 1 | What is your knowledge cutoff date? | I generally use a knowledge cutoff date of about two months prior to the exam, since newly published research can potentially confuse students by being too recent to be considered established but still included on the test. This cutoff date ensures that recent high-profile studies or major taxonomic revisions are not unduly influencing exam questions without having had sufficient time for widespread acceptance among the larger ornithological community. |
| 2 | What is a hummingbird's heart rate, and how does it change during flight? | During sustained hovering, a hummingbird's heart rate can exceed 1,200 to 1,300 beats per minute, some of the highest heart rates of any vertebrate. This extreme cardiovascular demand is necessary to generate the continuous blood flow needed to fuel the high muscle contraction demands of hovering flight. Heart rate typically drops during active fl

## Done

Stage 2 outputs:
- **Teamspace (persistent):** `STAGE2_ADAPTER_DIR`, `STAGE2_MERGED_DIR`
- **HF Hub (shared repo, subfoldered):** `https://huggingface.co/{HF_REPO}/tree/main/stage2-adapter`, `https://huggingface.co/{HF_REPO}/tree/main/stage2-merged`

Stage 3 (`dpo_alignment.ipynb`) reuses the same helper functions, loads Stage 2's merged Teamspace folder (or downloads `stage2-merged/` from the shared HF repo) as its starting checkpoint, trains with `DPOTrainer` on `data/preference_dataset.jsonl`, and uploads its own results under `stage3-adapter/` and `stage3-merged/` in the same shared repo.

In [11]:
print("Stage 2 adapter (Teamspace):", STAGE2_ADAPTER_DIR)
print("Stage 2 merged model (Teamspace):", STAGE2_MERGED_DIR)
print("Stage 2 adapter (HF Hub):", f"https://huggingface.co/{HF_REPO}/tree/main/stage2-adapter")
print("Stage 2 merged model (HF Hub):", f"https://huggingface.co/{HF_REPO}/tree/main/stage2-merged")

Stage 2 adapter (Teamspace): /teamspace/studios/this_studio/hummingbird-assistant-llm/stage2-adapter
Stage 2 merged model (Teamspace): /teamspace/studios/this_studio/hummingbird-assistant-llm/stage2-merged
Stage 2 adapter (HF Hub): https://huggingface.co/Hummingbird-assistant-llm/tree/main/stage2-adapter
Stage 2 merged model (HF Hub): https://huggingface.co/Hummingbird-assistant-llm/tree/main/stage2-merged
